In [7]:
import importlib
import numpy as np
import pandas as pd
from tir_reader import load_tir
import tyre_model as tm
importlib.reload(tm)

tyre = load_tir("data/sample.tir")
ref = pd.read_csv("data/mf61_reference.csv")

mine = {"Fx0": [], "Fy0": [], "Mz0": [], "Fx": [], "Fy": [], "Mz": []}

for _, row in ref.iterrows():
    a, k, Fz, g = row["alpha_rad"], row["kappa"], row["Fz"], row["gamma_rad"]
    mine["Fx0"].append(tm.longitudinal_force(tyre, k, Fz))
    mine["Fy0"].append(tm.lateral_force(tyre, a, Fz, g))
    mine["Mz0"].append(tm.aligning_moment(tyre, a, Fz, gamma=g)[0])
    Fx, Fy = tm.combined_force(tyre, k, a, Fz, g)
    mine["Fx"].append(Fx)
    mine["Fy"].append(Fy)
    mine["Mz"].append(tm.aligning_moment(tyre, a, Fz, kappa=k, gamma=g)[0])

for name, values in mine.items():
    error = np.abs(np.array(values) - ref[name].to_numpy())
    worst = np.argmax(error)
    print(f"{name}: max error {error[worst]:.3g}  at row {worst}")

Fx0: max error 4.8e-07  at row 480
Fy0: max error 15.8  at row 508
Mz0: max error 2.89  at row 498
Fx: max error 1.56e-06  at row 537
Fy: max error 16  at row 568
Mz: max error 3.04  at row 1862


In [8]:
print(ref.iloc[1821][["alpha_rad", "kappa", "Fz", "gamma_rad"]])
print("mine:", mine["Mz0"][1821], " reference:", ref["Mz0"][1821])

alpha_rad       0.000000
kappa          -1.000000
Fz           8000.000000
gamma_rad       0.069813
Name: 1821, dtype: float64
mine: -31.05738853428073  reference: -31.41236334


In [9]:
print(tm.lateral_force(tyre, np.radians(4), 4000))

-2879.6525418326237
